# LAB 6 — Análise de Erros de Faithfulness

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 5 — Diagnóstico e Correção de Falhas em Pipelines RAG**

---

## Objetivo

Identificar e diagnosticar queries com **Faithfulness < 0,70**, classificar causas raiz segundo a taxonomia de falhas e propor — e executar — correções específicas para cada causa.

---

## Taxonomia das 3 Causas de Baixo Faithfulness

| # | Causa | Definição | Sintoma Principal |
|---|-------|-----------|-------------------|
| 1 | **Alucinação Paramétrica** | O LLM usa conhecimento interno (pesos) em vez dos contextos fornecidos | Afirmações na resposta sem suporte em nenhum contexto recuperado |
| 2 | **Contexto Insuficiente** | Os trechos recuperados não cobrem as informações necessárias para responder | Grande parte da ground truth ausente nos contextos; Context Recall baixo |
| 3 | **Contexto Saturado** | Muitos contextos irrelevantes "diluem" os relevantes; o LLM confunde-se | Context Precision baixa; resposta mistura informações de trechos conflitantes |

## Diagrama de Decisão — Diagnóstico de Faithfulness Baixo

```
Faithfulness < 0.70?
        │
        ▼
Afirmações sem suporte nos contextos?
   ├─ SIM → Verificar: contextos cobrem a resposta?
   │           ├─ NÃO → CONTEXTO INSUFICIENTE (k maior / Contextual Retrieval)
   │           └─ SIM → ALUCINAÇÃO PARAMÉTRICA (prompt restritivo)
   └─ NÃO → Contextos conflitantes / irrelevantes?
               └─ SIM → CONTEXTO SATURADO (reranking top-3)
```

---

> **Norma:** ABNT NBR ISO/IEC 25010:2023 — Qualidade de Produto de Software  
> **Ambiente:** Google Colab • Python 3.11+ • GPU T4 • OpenSearch 3.x • vLLM (Llama 3.1 8B Instruct)

## 1. Instalação de Dependências

In [ ]:
# Instalação das dependências — execute apenas uma vez por sessão Colab
!pip install -q \
    ragas==0.1.21 \
    datasets==2.20.0 \
    langchain==0.2.16 \
    langchain-openai==0.1.25 \
    langchain-community==0.2.16 \
    sentence-transformers==3.1.1 \
    opensearch-py==2.7.1 \
    pandas==2.2.2 \
    matplotlib==3.9.2 \
    numpy==1.26.4

print("✓ Dependências instaladas com sucesso.")

## 2. Importações e Configuração de Variáveis de Ambiente

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# Variáveis de ambiente — ajuste conforme seu ambiente de lab
# ─────────────────────────────────────────────────────────────
os.environ["VLLM_BASE_URL"]       = os.getenv("VLLM_BASE_URL",       "http://localhost:8000/v1")
os.environ["VLLM_MODEL_NAME"]     = os.getenv("VLLM_MODEL_NAME",     "meta-llama/Llama-3.1-8B-Instruct")
os.environ["VLLM_API_KEY"]        = os.getenv("VLLM_API_KEY",        "token-lab-mba-2025")
os.environ["OPENSEARCH_HOST"]     = os.getenv("OPENSEARCH_HOST",     "localhost")
os.environ["OPENSEARCH_PORT"]     = os.getenv("OPENSEARCH_PORT",     "9200")
os.environ["OPENSEARCH_INDEX"]    = os.getenv("OPENSEARCH_INDEX",    "corpus_juridico")
os.environ["LANGFUSE_HOST"]       = os.getenv("LANGFUSE_HOST",       "http://localhost:3000")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-mba-lab6")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-mba-lab6")
os.environ["EMBEDDER_MODEL"]      = os.getenv("EMBEDDER_MODEL",      "BAAI/bge-m3")
os.environ["RERANKER_MODEL"]      = os.getenv("RERANKER_MODEL",      "BAAI/bge-reranker-v2-m3")

# Caminho do arquivo de baseline
BASELINE_PATH = "ragas_baseline_resultados.csv"
DATASET_PATH  = "dataset_avaliacao_completo.json"

# Limiar de Faithfulness para análise de erros
LIMIAR_FAITHFULNESS = 0.70

print(f"✓ Variáveis de ambiente configuradas.")
print(f"  vLLM:       {os.environ['VLLM_BASE_URL']}")
print(f"  OpenSearch: {os.environ['OPENSEARCH_HOST']}:{os.environ['OPENSEARCH_PORT']}")
print(f"  LangFuse:   {os.environ['LANGFUSE_HOST']}")
print(f"  Limiar de análise: Faithfulness < {LIMIAR_FAITHFULNESS}")

## 3. Health Checks de Conexões

In [ ]:
import requests
from opensearchpy import OpenSearch
from sentence_transformers import SentenceTransformer

STATUS = {}

# ── Health Check: vLLM ──────────────────────────────────────
try:
    resp = requests.get(
        f"{os.environ['VLLM_BASE_URL'].rstrip('/v1')}/health",
        timeout=5
    )
    STATUS["vLLM"] = "✅ OK" if resp.status_code == 200 else f"⚠ HTTP {resp.status_code}"
except Exception as e:
    STATUS["vLLM"] = f"❌ Offline ({e})"

# ── Health Check: OpenSearch ────────────────────────────────
try:
    os_client = OpenSearch(
        hosts=[{"host": os.environ["OPENSEARCH_HOST"],
                "port": int(os.environ["OPENSEARCH_PORT"])}],
        http_compress=True,
        use_ssl=False,
        verify_certs=False,
        timeout=5,
    )
    info = os_client.info()
    STATUS["OpenSearch"] = f"✅ OK — versão {info['version']['number']}"
except Exception as e:
    os_client = None
    STATUS["OpenSearch"] = f"❌ Offline ({e})"

# ── Health Check: LangFuse ───────────────────────────────────
try:
    resp = requests.get(
        f"{os.environ['LANGFUSE_HOST']}/api/public/health",
        timeout=5
    )
    STATUS["LangFuse"] = "✅ OK" if resp.status_code == 200 else f"⚠ HTTP {resp.status_code}"
except Exception as e:
    STATUS["LangFuse"] = f"❌ Offline ({e})"

# ── Health Check: Embedder BGE-M3 ────────────────────────────
try:
    embedder = SentenceTransformer(os.environ["EMBEDDER_MODEL"])
    _ = embedder.encode(["saúde"])
    STATUS["Embedder"] = f"✅ OK — {os.environ['EMBEDDER_MODEL']}"
except Exception as e:
    embedder = None
    STATUS["Embedder"] = f"❌ Falhou ({e})"

# ── Exibir resumo ────────────────────────────────────────────
print("=" * 55)
print("         HEALTH CHECK — LAB 6")
print("=" * 55)
for srv, st in STATUS.items():
    print(f"  {srv:<15} {st}")
print("=" * 55)

## 4. Carregar Baseline do LAB 2 e Dataset de Avaliação

In [ ]:
def criar_dataset_sintetico_lab6(n: int = 50) -> tuple[pd.DataFrame, list[dict]]:
    """
    Cria DataFrame simulado com ~30% dos pares com Faithfulness < 0.70.
    Os casos de baixo Faithfulness são distribuídos entre as 3 causas.
    Retorna (df_scores, lista_pares_completos).
    """
    np.random.seed(42)

    # Perguntas jurídicas com diferentes graus de complexidade
    perguntas = [
        "Quais são os requisitos para prisão preventiva no CPP?",
        "Como se aplica o princípio da proporcionalidade em buscas policiais?",
        "O que caracteriza crime de responsabilidade de servidor público?",
        "Quais direitos o preso tem durante a audiência de custódia?",
        "Quando cabe habeas corpus preventivo segundo o STF?",
        "Quais são os elementos do crime de peculato?",
        "Como funciona a delação premiada na Lei 12.850/2013?",
        "Quais são as hipóteses de extinção da punibilidade?",
        "O que é o princípio da ultima ratio no direito penal?",
        "Como se configura o abuso de autoridade policial?",
        "Quais são os requisitos para decretação de interceptação telefônica?",
        "O que é o relaxamento de prisão ilegal?",
        "Quando se aplica a Lei Maria da Penha em casos policiais?",
        "Quais são as prerrogativas do advogado em delegacia?",
        "Como se configura o crime de desacato à autoridade?",
        "O que é a teoria da cegueira deliberada no direito penal?",
        "Quais são as espécies de medidas cautelares diversas da prisão?",
        "Como funciona o princípio da dignidade da pessoa humana na segurança pública?",
        "Quais são os limites constitucionais do poder de polícia?",
        "O que caracteriza a flagrância delitiva?",
        "Quando é possível decretar sigilo em investigações policiais?",
        "Quais são as garantias do acusado no processo penal brasileiro?",
        "Como se aplica o princípio do contraditório nas investigações?",
        "Quais são as competências do delegado de polícia federal?",
        "O que é inquérito policial e qual sua natureza jurídica?",
        "Quais são os fundamentos para prisão em flagrante?",
        "Como funciona a liberdade provisória com fiança?",
        "O que é o mandado de segurança coletivo em matéria policial?",
        "Quais são as hipóteses de uso de força pelo policial?",
        "Como se configura a legítima defesa no exercício policial?",
        "Quais são os crimes hediondos e suas consequências processuais?",
        "Como funciona o sistema de progressão de regime penitenciário?",
        "O que é o princípio da legalidade estrita no direito penal?",
        "Quais são os direitos do preso na Lei de Execução Penal?",
        "Como se realiza o reconhecimento de pessoas no processo penal?",
        "Quais são as fases do procedimento do júri popular?",
        "O que é a ação penal de iniciativa privada subsidiária?",
        "Quais são os requisitos para o acordo de não persecução penal?",
        "Como funciona a prova pericial no processo penal?",
        "O que é o princípio da presunção de inocência?",
        "Quais são as hipóteses de absolvição sumária?",
        "Como se configura o crime de tráfico de drogas equiparado a hediondo?",
        "O que é a colaboração premiada e seus requisitos?",
        "Quais são os prazos da investigação preliminar no CPP?",
        "Como funciona o controle externo da atividade policial pelo MP?",
        "O que é o inquérito civil e sua relação com a segurança pública?",
        "Quais são os princípios do uso da força pela ONU aplicáveis no Brasil?",
        "Como se configura o crime de tortura praticado por agente público?",
        "O que é o habeas data e quando cabe em matéria de segurança?",
        "Quais são as regras de uso de algemas segundo o STF?",
    ]

    # Índices que terão Faithfulness < 0.70 (~30%)
    indices_baixos = [2, 5, 7, 11, 14, 16, 19, 22, 25, 28, 31, 33, 36, 39, 44]

    # Causas simuladas para os casos de baixo Faithfulness
    causas = [
        "alucinacao_parametrica", "contexto_insuficiente", "contexto_saturado",
        "alucinacao_parametrica", "contexto_insuficiente", "contexto_saturado",
        "alucinacao_parametrica", "contexto_insuficiente", "contexto_saturado",
        "alucinacao_parametrica", "contexto_insuficiente", "contexto_saturado",
        "alucinacao_parametrica", "contexto_insuficiente", "contexto_saturado",
    ]

    # Gerar scores
    faithfulness_scores = np.clip(np.random.normal(0.75, 0.07, n), 0.60, 0.95)
    for idx in indices_baixos:
        if idx < n:
            faithfulness_scores[idx] = np.random.uniform(0.40, 0.68)

    df = pd.DataFrame({
        "id":                 range(1, n + 1),
        "question":           perguntas[:n],
        "faithfulness":       faithfulness_scores.round(4),
        "answer_relevancy":   np.clip(np.random.normal(0.70, 0.07, n), 0.50, 0.93).round(4),
        "context_recall":     np.clip(np.random.normal(0.67, 0.08, n), 0.42, 0.92).round(4),
        "context_precision":  np.clip(np.random.normal(0.68, 0.08, n), 0.42, 0.92).round(4),
        "causa_simulada":     [""] * n,
    })

    for i, idx in enumerate(indices_baixos):
        if idx < n:
            df.at[idx, "causa_simulada"] = causas[i]

    # Criar pares completos com contextos e respostas
    pares = []
    for _, row in df.iterrows():
        causa = row["causa_simulada"]
        # Simular contextos e respostas de acordo com a causa
        if causa == "alucinacao_parametrica":
            answer = (
                f"Com base nos meus conhecimentos, o Art. 312 do CPP prevê que a prisão preventiva "
                f"pode ser decretada para garantir a ordem pública. Além disso, o STJ consolidou "
                f"entendimento de que a reincidência é fundamento autônomo para preventiva — "
                f"informação presente nos meus dados de treinamento, não nos trechos fornecidos."
            )
            contexts = [
                "Art. 311 do CPP: Em qualquer fase da investigação policial, o juiz poderá decretar prisão preventiva.",
                "Art. 282 do CPP: As medidas cautelares serão decretadas pelo juiz competente.",
                "Lei 7.960/1989: Trata da prisão temporária como medida cautelar diversa.",
            ]
        elif causa == "contexto_insuficiente":
            answer = (
                f"A delação premiada é regulada pela Lei 12.850/2013. O colaborador deve cumprir "
                f"os requisitos do art. 4º, incluindo voluntariedade, eficácia e proporcionalidade. "
                f"O Ministério Público pode propor o acordo em qualquer fase do inquérito ou processo."
            )
            contexts = [
                "Lei 12.850/2013 — Organizações Criminosas: Define os crimes praticados por organização criminosa.",
                "Art. 1º da Lei 12.850/2013: Considera-se organização criminosa a associação de 4 ou mais pessoas.",
            ]
        elif causa == "contexto_saturado":
            answer = (
                f"O crime de peculato está previsto no art. 312 do Código Penal. "
                f"O servidor que se apropria de dinheiro público pratica peculato-apropriação. "
                f"O prazo prescricional segue as regras gerais do art. 109 do CP, combinado com "
                f"normas tributárias e administrativas que também tratam de responsabilidade funcional."
            )
            contexts = [
                "Art. 312 do CP: Apropriação ou desvio de dinheiro ou bem de que o servidor tem a guarda.",
                "Art. 312, §1º: Se o servidor facilitar a subtração por outeiro, a pena é a do peculato.",
                "Art. 315 do CP: Emprego irregular de verbas ou rendas públicas.",
                "Lei 8.429/1992: Improbidade administrativa — arts. de enriquecimento ilícito.",
                "Lei 9.784/1999: Processo administrativo federal — art. 2º — princípios.",
                "Decreto-Lei 200/1967: Organização da Administração Federal — art. 25 — controle.",
                "CF/1988, art. 37, §4º: Improbidade acarretará suspensão de direitos políticos.",
                "STJ Súmula 599: O princípio da insignificância é inaplicável ao crime de peculato.",
                "Lei 10.028/2000: Altera o CP para incluir crimes de responsabilidade fiscal.",
                "Art. 109 do CP: Prescrição — antes de transitar em julgado a sentença final.",
            ]
        else:
            # Par sem problema de faithfulness
            answer = (
                f"Com base nos trechos do corpus jurídico, a resposta à questão "
                f"'{row['question'][:50]}...' é construída a partir das normas citadas."
            )
            contexts = [
                f"Trecho A referente à questão {int(row['id'])}: norma jurídica pertinente.",
                f"Trecho B referente à questão {int(row['id'])}: doutrina aplicável.",
            ]

        pares.append({
            "id":              int(row["id"]),
            "question":        row["question"],
            "answer":          answer,
            "contexts":        contexts,
            "ground_truth":    f"Resposta de referência — questão {int(row['id'])}.",
            "faithfulness":    float(row["faithfulness"]),
            "causa_simulada":  causa,
        })

    return df, pares


# ── Carregar ou criar baseline ───────────────────────────────
if os.path.exists(BASELINE_PATH):
    df_baseline = pd.read_csv(BASELINE_PATH)
    if "id" not in df_baseline.columns:
        df_baseline.insert(0, "id", range(1, len(df_baseline) + 1))
    print(f"✓ Baseline carregado: {BASELINE_PATH} ({len(df_baseline)} pares)")

    # Criar pares completos a partir do dataset do LAB1, se disponível
    if os.path.exists(DATASET_PATH):
        with open(DATASET_PATH, "r", encoding="utf-8") as f:
            dataset_completo = json.load(f)
        # Mesclar scores do CSV com dados completos do JSON
        pares_completos = []
        for i, par in enumerate(dataset_completo[:len(df_baseline)]):
            row = df_baseline.iloc[i]
            pares_completos.append({
                **par,
                "faithfulness":   float(row.get("faithfulness", 0.75)),
                "causa_simulada": "",
            })
    else:
        # Fallback: criar pares sintéticos
        _, pares_completos = criar_dataset_sintetico_lab6(len(df_baseline))
else:
    print(f"⚠ '{BASELINE_PATH}' não encontrado. Criando dados simulados (~30% com Faithfulness < 0.70)...")
    df_baseline, pares_completos = criar_dataset_sintetico_lab6(n=50)
    df_baseline.to_csv(BASELINE_PATH, index=False)
    print(f"  → Baseline simulado salvo em '{BASELINE_PATH}'.")

print(f"\nTotal de pares carregados: {len(df_baseline)}")
print(f"Colunas disponíveis: {list(df_baseline.columns)}")

## 5. Filtrar Queries com Faithfulness < 0,70

In [ ]:
# ── Filtrar pares com Faithfulness abaixo do limiar ──────────
df_baixo = df_baseline[df_baseline["faithfulness"] < LIMIAR_FAITHFULNESS].copy()
df_baixo = df_baixo.sort_values("faithfulness").reset_index(drop=True)

n_total = len(df_baseline)
n_baixo = len(df_baixo)
pct_baixo = (n_baixo / n_total) * 100

print("=" * 60)
print(f"  QUERIES COM FAITHFULNESS < {LIMIAR_FAITHFULNESS}")
print("=" * 60)
print(f"  Total de pares avaliados : {n_total}")
print(f"  Com Faithfulness < {LIMIAR_FAITHFULNESS}   : {n_baixo} ({pct_baixo:.1f}%)")
print(f"  Com Faithfulness >= {LIMIAR_FAITHFULNESS}  : {n_total - n_baixo} ({100-pct_baixo:.1f}%)")
print("=" * 60)

# Exibir os casos de baixo faithfulness (coluna question abreviada)
print("\nQuerys com Faithfulness abaixo do limiar (ordenadas do menor ao maior score):")
cols_exibir = ["id", "faithfulness", "question"] if "question" in df_baixo.columns else ["id", "faithfulness"]
df_exibir = df_baixo[cols_exibir].copy()
if "question" in df_exibir.columns:
    df_exibir["question"] = df_exibir["question"].str[:60] + "..."
print(df_exibir.to_string(index=False))

## 6. Função `diagnosticar_query` — Diagnóstico via vLLM

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

# ── Instanciar LLM ──────────────────────────────────────────
try:
    llm = ChatOpenAI(
        base_url=os.environ["VLLM_BASE_URL"],
        api_key=os.environ["VLLM_API_KEY"],
        model=os.environ["VLLM_MODEL_NAME"],
        temperature=0.0,
        max_tokens=1024,
        timeout=60,
    )
    LLM_DISPONIVEL = True
    print("✓ LLM (vLLM) instanciado.")
except Exception as e:
    llm = None
    LLM_DISPONIVEL = False
    print(f"⚠ LLM indisponível: {e} — usando diagnóstico baseado em heurísticas.")


CAUSAS_VALIDAS = {
    "alucinacao_parametrica": "Alucinação Paramétrica",
    "contexto_insuficiente":  "Contexto Insuficiente",
    "contexto_saturado":      "Contexto Saturado",
}


def diagnosticar_por_heuristica(
    question: str,
    answer: str,
    contexts: list[str],
    faithfulness_score: float
) -> dict:
    """
    Diagnóstico heurístico quando o LLM não está disponível.
    Usa sinais léxicos e tamanho dos contextos para classificar.
    """
    # Heurística 1: se há muitos contextos (>= 6) → Contexto Saturado
    if len(contexts) >= 6:
        causa = "contexto_saturado"
        afirmacoes = [answer[:100]]
        sem_suporte = [answer[:100]]
        explicacao = (
            f"{len(contexts)} contextos recuperados — excesso de trechos possivelmente "
            "não relacionados dilui a relevância e confunde o LLM."
        )
    # Heurística 2: contextos muito curtos ou poucos → Contexto Insuficiente
    elif len(contexts) <= 2 or all(len(c) < 100 for c in contexts):
        causa = "contexto_insuficiente"
        afirmacoes = [answer[:100]]
        sem_suporte = [answer[:100]]
        explicacao = (
            f"Apenas {len(contexts)} contexto(s) recuperado(s), com cobertura insuficiente "
            "para suportar todas as afirmações da resposta."
        )
    # Heurística 3: por eliminação → Alucinação Paramétrica
    else:
        causa = "alucinacao_parametrica"
        afirmacoes = [answer[:100]]
        sem_suporte = [answer[:100]]
        # Verificar marcadores linguísticos de alucinação
        marcadores = ["meus conhecimentos", "treinamento", "sabe-se que",
                      "é amplamente conhecido", "conforme estudos"]
        marcadores_presentes = [m for m in marcadores if m.lower() in answer.lower()]
        explicacao = (
            f"LLM usou conhecimento paramétrico (pesos) em vez dos contextos. "
            f"Marcadores detectados: {marcadores_presentes if marcadores_presentes else 'nenhum léxico explícito, mas afirmações não rastreáveis'}."
        )

    return {
        "questao":                question,
        "faithfulness_original":  faithfulness_score,
        "afirmacoes_na_resposta": afirmacoes,
        "afirmacoes_sem_suporte": sem_suporte,
        "causa":                  causa,
        "causa_legivel":          CAUSAS_VALIDAS[causa],
        "explicacao":             explicacao,
        "metodo_diagnostico":     "heuristica",
    }


def diagnosticar_query(
    question: str,
    answer: str,
    contexts: list[str],
    faithfulness_score: float
) -> dict:
    """
    Diagnostica uma query com Faithfulness baixo usando o LLM.

    Fluxo:
      (a) Lista as afirmações da resposta
      (b) Verifica quais afirmações estão suportadas pelos contextos
      (c) Classifica a causa: alucinacao_parametrica / contexto_insuficiente / contexto_saturado
      (d) Retorna dict com diagnóstico detalhado

    Se o LLM estiver indisponível, usa heurísticas estruturais.
    """
    if not LLM_DISPONIVEL or llm is None:
        return diagnosticar_por_heuristica(question, answer, contexts, faithfulness_score)

    contextos_texto = "\n".join(
        [f"[Contexto {i+1}]: {c[:300]}" for i, c in enumerate(contexts)]
    )

    # Prompt de diagnóstico estruturado
    prompt = f"""Você é um especialista em avaliação de sistemas RAG (Retrieval-Augmented Generation) para o domínio jurídico.

PERGUNTA ORIGINAL: {question}

RESPOSTA GERADA PELO RAG:
{answer}

CONTEXTOS RECUPERADOS:
{contextos_texto}

FAITHFULNESS MEDIDO PELO RAGAS: {faithfulness_score:.3f} (abaixo do limiar de 0.70)

TAREFA:
1. Liste as principais afirmações feitas na resposta (máximo 5, uma por linha).
2. Para cada afirmação, indique se está (SIM) ou não está (NÃO) suportada pelos contextos.
3. Classifique a causa do baixo Faithfulness em EXATAMENTE UMA das categorias abaixo:
   - alucinacao_parametrica: O LLM usou conhecimento interno (pesos) sem base nos contextos
   - contexto_insuficiente: Os contextos recuperados não cobrem o que era necessário para responder
   - contexto_saturado: Excesso de contextos irrelevantes confundiu o LLM
4. Explique em 2 frases por que classificou assim.

RESPONDA NESTE FORMATO JSON:
{{
  "afirmacoes": ["afirmação 1", "afirmação 2"],
  "suporte": ["SIM", "NÃO"],
  "causa": "alucinacao_parametrica" | "contexto_insuficiente" | "contexto_saturado",
  "explicacao": "Explicação em 2 frases."
}}"""

    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        texto = resp.content.strip()

        # Extrair JSON da resposta
        inicio = texto.find("{")
        fim    = texto.rfind("}") + 1
        dados  = json.loads(texto[inicio:fim])

        # Validar causa
        causa = dados.get("causa", "alucinacao_parametrica")
        if causa not in CAUSAS_VALIDAS:
            causa = "alucinacao_parametrica"

        afirmacoes   = dados.get("afirmacoes", [])
        suporte      = dados.get("suporte", [])
        sem_suporte  = [a for a, s in zip(afirmacoes, suporte) if s.upper() == "NÃO"]

        return {
            "questao":                question,
            "faithfulness_original":  faithfulness_score,
            "afirmacoes_na_resposta": afirmacoes,
            "afirmacoes_sem_suporte": sem_suporte,
            "causa":                  causa,
            "causa_legivel":          CAUSAS_VALIDAS[causa],
            "explicacao":             dados.get("explicacao", ""),
            "metodo_diagnostico":     "llm",
        }

    except Exception as e:
        print(f"  ⚠ Falha no diagnóstico LLM: {e} — usando heurística.")
        return diagnosticar_por_heuristica(question, answer, contexts, faithfulness_score)


print("✓ Função diagnosticar_query definida.")

## 7. Diagnóstico das 3 Queries com Faithfulness Mais Baixo

In [ ]:
# ── Selecionar as 3 piores queries ───────────────────────────
df_top3_baixo = df_baixo.head(3).copy()

print("3 queries com Faithfulness mais baixo:")
for _, row in df_top3_baixo.iterrows():
    q = row.get("question", f"Query ID {int(row['id'])}")[:70]
    print(f"  ID={int(row['id'])}  Faith={row['faithfulness']:.4f}  Q: {q}...")

# ── Mapear IDs para pares completos ──────────────────────────
mapa_pares = {p["id"]: p for p in pares_completos}

# ── Executar diagnóstico ─────────────────────────────────────
diagnosticos = []

for i, row in df_top3_baixo.iterrows():
    id_query = int(row["id"])
    par      = mapa_pares.get(id_query, {})

    question  = par.get("question",  row.get("question",  f"Query {id_query}"))
    answer    = par.get("answer",    "[resposta não disponível]")
    contexts  = par.get("contexts",  ["[contexto não disponível]"])
    faith     = float(row["faithfulness"])

    print(f"\n{'='*60}")
    print(f"DIAGNÓSTICO — Query ID {id_query} (Faithfulness={faith:.4f})")
    print(f"{'='*60}")
    print(f"Pergunta: {question[:80]}...")
    print(f"Contextos disponíveis: {len(contexts)}")
    print("Executando diagnóstico...", end=" ")

    diag = diagnosticar_query(
        question=question,
        answer=answer,
        contexts=contexts,
        faithfulness_score=faith,
    )
    diagnosticos.append(diag)

    print("✓")
    print(f"\n  Causa classificada : {diag['causa_legivel']}")
    print(f"  Método de diagnóstico: {diag['metodo_diagnostico']}")
    print(f"  Afirmações na resposta: {len(diag['afirmacoes_na_resposta'])}")
    print(f"  Afirmações sem suporte nos contextos: {len(diag['afirmacoes_sem_suporte'])}")
    print(f"  Explicação: {diag['explicacao']}")

print(f"\n✓ Diagnóstico concluído para {len(diagnosticos)} queries.")

## 8. Aplicar Correções por Causa e Recalcular Faithfulness

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness as ragas_faithfulness
from datasets import Dataset as HFDataset


def calcular_faithfulness_simples(question: str, answer: str, contexts: list[str]) -> float:
    """
    Calcula Faithfulness via RAGAS para um único par.
    Se RAGAS falhar, usa estimativa heurística de cobertura léxica.
    """
    try:
        from langchain_openai import ChatOpenAI as LC_Chat
        from langchain_openai import OpenAIEmbeddings

        ragas_llm = LC_Chat(
            base_url=os.environ["VLLM_BASE_URL"],
            api_key=os.environ["VLLM_API_KEY"],
            model=os.environ["VLLM_MODEL_NAME"],
            temperature=0,
        )
        ragas_emb = OpenAIEmbeddings(
            base_url=os.environ["VLLM_BASE_URL"],
            api_key=os.environ["VLLM_API_KEY"],
            model=os.environ["EMBEDDER_MODEL"],
        )
        ds = HFDataset.from_list([{
            "question":     question,
            "answer":       answer,
            "contexts":     contexts,
            "ground_truth": "",
        }])
        resultado = evaluate(
            dataset=ds,
            metrics=[ragas_faithfulness],
            llm=ragas_llm,
            embeddings=ragas_emb,
        )
        return float(resultado.to_pandas()["faithfulness"].iloc[0])
    except Exception:
        # Heurística: proporção de palavras da resposta encontradas nos contextos
        contexto_total = " ".join(contexts).lower()
        palavras_resposta = set(answer.lower().split())
        palavras_stopwords = {"a", "o", "e", "de", "da", "do", "em", "que",
                              "se", "na", "no", "os", "as", "um", "uma", "para", "com"}
        palavras_relevantes = palavras_resposta - palavras_stopwords
        if not palavras_relevantes:
            return 0.75  # fallback neutro
        cobertura = sum(1 for p in palavras_relevantes if p in contexto_total)
        return min(round(cobertura / len(palavras_relevantes), 3), 0.98)


def corrigir_alucinacao_parametrica(
    question: str, contexts: list[str]
) -> str:
    """Correção: prompt restritivo — LLM instruído a usar APENAS os trechos fornecidos."""
    contexto_texto = "\n\n".join(
        [f"Trecho {i+1}:\n{ctx}" for i, ctx in enumerate(contexts)]
    )
    prompt = (
        f"INSTRUÇÃO ESTRITA: Use APENAS os trechos abaixo para responder. "
        f"Se a informação não estiver nos trechos, responda: "
        f"'As informações disponíveis nos documentos recuperados não são suficientes para responder a esta questão.'\n\n"
        f"TRECHOS DO CORPUS JURÍDICO:\n{contexto_texto}\n\n"
        f"PERGUNTA: {question}\n\n"
        f"RESPOSTA (baseada exclusivamente nos trechos acima):"
    )
    if llm is None:
        return (
            f"[Resposta corrigida — prompt restritivo] Com base exclusivamente nos "
            f"{len(contexts)} trechos fornecidos: {contexts[0][:150]}... "
            f"Sem extrapolar além do material recuperado."
        )
    try:
        resp = llm.invoke([HumanMessage(content=prompt)])
        return resp.content.strip()
    except Exception as e:
        return f"[Erro na correção: {e}]"


def corrigir_contexto_insuficiente(
    question: str, k_extra: int = 10
) -> tuple[str, list[str]]:
    """Correção: k maior (k=10) + prompt de Contextual Retrieval."""
    # Recuperar mais contextos
    if os_client is not None and embedder is not None:
        try:
            vetor = embedder.encode([question])[0].tolist()
            resp_os = os_client.search(
                index=os.environ["OPENSEARCH_INDEX"],
                body={"size": k_extra, "query": {"knn": {"embedding": {"vector": vetor, "k": k_extra}}}}
            )
            contextos_ampliados = [
                h["_source"].get("content", h["_source"].get("text", ""))
                for h in resp_os["hits"]["hits"]
            ]
        except Exception:
            contextos_ampliados = [
                f"[Contexto ampliado {i+1}] Trecho adicional recuperado com k={k_extra} para '{question[:40]}'"
                for i in range(k_extra)
            ]
    else:
        contextos_ampliados = [
            f"[Contexto ampliado {i+1}] Trecho adicional do corpus jurídico — lei, norma ou doutrina aplicável."
            for i in range(k_extra)
        ]

    contexto_texto = "\n\n".join(
        [f"Trecho {i+1}:\n{ctx}" for i, ctx in enumerate(contextos_ampliados[:5])]
    )
    prompt = (
        f"Com base nos trechos jurídicos abaixo (recuperados com cobertura ampliada), "
        f"responda de forma completa e precisa.\n\n"
        f"TRECHOS:\n{contexto_texto}\n\n"
        f"PERGUNTA: {question}\n\n"
        f"RESPOSTA:"
    )
    if llm is None:
        resposta = (
            f"[Resposta corrigida — k ampliado] Com base nos {k_extra} trechos recuperados, "
            f"a cobertura do corpus foi ampliada para: {contextos_ampliados[0][:120]}..."
        )
    else:
        try:
            resp = llm.invoke([HumanMessage(content=prompt)])
            resposta = resp.content.strip()
        except Exception as e:
            resposta = f"[Erro: {e}]"

    return resposta, contextos_ampliados[:5]


def corrigir_contexto_saturado(
    question: str, contextos: list[str]
) -> tuple[str, list[str]]:
    """Correção: reranking para selecionar apenas top-3 contextos mais relevantes."""
    # Reranking por similaridade coseno (fallback se CrossEncoder indisponível)
    if embedder is not None and len(contextos) > 3:
        from numpy.linalg import norm
        q_emb = embedder.encode([question])[0]
        c_embs = embedder.encode(contextos)
        scores = [
            float(np.dot(q_emb, c) / (norm(q_emb) * norm(c) + 1e-9))
            for c in c_embs
        ]
        top3_idx = np.argsort(scores)[::-1][:3]
        contextos_filtrados = [contextos[i] for i in top3_idx]
    else:
        contextos_filtrados = contextos[:3]

    contexto_texto = "\n\n".join(
        [f"Trecho {i+1}:\n{ctx}" for i, ctx in enumerate(contextos_filtrados)]
    )
    prompt = (
        f"Use APENAS os 3 trechos mais relevantes abaixo para responder com precisão.\n\n"
        f"TRECHOS SELECIONADOS (pós-reranking):\n{contexto_texto}\n\n"
        f"PERGUNTA: {question}\n\n"
        f"RESPOSTA:"
    )
    if llm is None:
        resposta = (
            f"[Resposta corrigida — reranking aplicado] Usando os 3 trechos mais relevantes: "
            f"{contextos_filtrados[0][:120]}..."
        )
    else:
        try:
            resp = llm.invoke([HumanMessage(content=prompt)])
            resposta = resp.content.strip()
        except Exception as e:
            resposta = f"[Erro: {e}]"

    return resposta, contextos_filtrados


# ── Aplicar correção para cada caso diagnosticado ────────────
resultados_correcao = []

for i, diag in enumerate(diagnosticos):
    id_query = df_top3_baixo.iloc[i]["id"]
    par      = mapa_pares.get(int(id_query), {})

    question        = diag["questao"]
    causa           = diag["causa"]
    faith_antes     = diag["faithfulness_original"]
    contextos_orig  = par.get("contexts", ["[sem contexto]"])

    print(f"\n{'─'*60}")
    print(f"Correção [{i+1}/3] — Causa: {diag['causa_legivel']}")
    print(f"  Query: {question[:65]}...")
    print(f"  Faithfulness antes: {faith_antes:.4f}")

    # Aplicar a correção correspondente à causa
    if causa == "alucinacao_parametrica":
        print("  Estratégia: Prompt restritivo — 'Use APENAS os trechos abaixo'")
        answer_corrigida = corrigir_alucinacao_parametrica(question, contextos_orig)
        contextos_finais = contextos_orig
        correcao_aplicada = "Prompt restritivo"

    elif causa == "contexto_insuficiente":
        print("  Estratégia: k=10 com Contextual Retrieval")
        answer_corrigida, contextos_finais = corrigir_contexto_insuficiente(question, k_extra=10)
        correcao_aplicada = "k=10 + Contextual Retrieval"

    elif causa == "contexto_saturado":
        print("  Estratégia: Reranking → top-3 mais relevantes")
        answer_corrigida, contextos_finais = corrigir_contexto_saturado(question, contextos_orig)
        correcao_aplicada = "Reranking top-3"

    else:
        answer_corrigida = par.get("answer", "")
        contextos_finais = contextos_orig
        correcao_aplicada = "Nenhuma"

    # Recalcular Faithfulness após correção
    print("  Calculando Faithfulness após correção...", end=" ")
    faith_depois = calcular_faithfulness_simples(question, answer_corrigida, contextos_finais)
    print(f"✓  {faith_depois:.4f}")

    melhoria = faith_depois - faith_antes
    print(f"  Melhoria: {melhoria:+.4f} ({(melhoria/faith_antes)*100:+.1f}%)")

    resultados_correcao.append({
        "query":             question[:60] + "...",
        "causa":             diag["causa_legivel"],
        "faith_antes":       round(faith_antes, 4),
        "faith_depois":      round(faith_depois, 4),
        "melhoria":          round(melhoria, 4),
        "correcao_aplicada": correcao_aplicada,
    })

print("\n✓ Correções aplicadas e Faithfulness recalculado para os 3 casos.")

## 9. Tabela Comparativa — Antes vs. Depois da Correção

In [ ]:
# ── Montar tabela comparativa ────────────────────────────────
df_correcao = pd.DataFrame(resultados_correcao)

print("=" * 85)
print("  TABELA COMPARATIVA — FAITHFULNESS ANTES E DEPOIS DA CORREÇÃO")
print("=" * 85)
print(df_correcao.to_string(index=False))
print("=" * 85)

# Exportar tabela
df_correcao.to_csv("analise_erros_faithfulness.csv", index=False)
print("\n✓ Tabela exportada: analise_erros_faithfulness.csv")

## 10. Gráfico: Faithfulness Antes vs. Depois (Barras Horizontais)

In [ ]:
# ── Configuração do gráfico ──────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#F8F9FA",
    "axes.facecolor":   "#FFFFFF",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.axis":        "x",
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

fig, ax = plt.subplots(figsize=(11, 5), facecolor="#F8F9FA")

n_casos   = len(df_correcao)
y_pos     = np.arange(n_casos)
altura    = 0.32

# Rótulos das queries (abreviados)
labels_query = [
    f"Q{i+1}: {row['query'][:45]}..."
    for i, (_, row) in enumerate(df_correcao.iterrows())
]

# Barras: azul = antes, verde = depois
barras_antes  = ax.barh(y_pos + altura/2, df_correcao["faith_antes"],  altura,
                         color="#4472C4", alpha=0.85, label="Faithfulness (antes)",
                         edgecolor="white")
barras_depois = ax.barh(y_pos - altura/2, df_correcao["faith_depois"], altura,
                         color="#70AD47", alpha=0.85, label="Faithfulness (depois)",
                         edgecolor="white")

# Linha vertical: limiar 0.70
ax.axvline(x=LIMIAR_FAITHFULNESS, color="#FF0000", linestyle="--",
           linewidth=1.8, label=f"Limiar {LIMIAR_FAITHFULNESS}")

# Anotações nas barras
for barra in barras_antes:
    w = barra.get_width()
    ax.text(w + 0.005, barra.get_y() + barra.get_height()/2,
            f"{w:.3f}", va="center", ha="left", fontsize=9,
            color="#4472C4", fontweight="bold")
for barra in barras_depois:
    w = barra.get_width()
    ax.text(w + 0.005, barra.get_y() + barra.get_height()/2,
            f"{w:.3f}", va="center", ha="left", fontsize=9,
            color="#375623", fontweight="bold")

# Anotação de causa em cada caso
for i, (_, row) in enumerate(df_correcao.iterrows()):
    ax.text(0.01, y_pos[i], row["causa"],
            va="center", ha="left", fontsize=7.5,
            color="#6B7280", style="italic")

ax.set_yticks(y_pos)
ax.set_yticklabels(labels_query, fontsize=8.5)
ax.set_xlim(0, 1.05)
ax.set_xlabel("Score Faithfulness", fontsize=10)
ax.set_title(
    "LAB 6 — Faithfulness Antes e Depois da Correção\n"
    "(3 queries com menor score — Naive RAG Baseline)",
    fontsize=12, fontweight="bold", pad=10
)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig("grafico_correcao_faithfulness.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✓ Gráfico salvo: grafico_correcao_faithfulness.png")

## 11. Distribuição de Causas — Pie Chart

In [ ]:
# ── Contagem de causas nos casos analisados ──────────────────
# Usar todos os casos de baixo faithfulness, não só os 3 exemplos
# Se causa_simulada disponível, usar; caso contrário, inferir dos diagnósticos

causas_contagem = {"Alucinação Paramétrica": 0, "Contexto Insuficiente": 0, "Contexto Saturado": 0}

# Contar a partir dos diagnósticos dos 3 casos analisados (extrapolação)
for diag in diagnosticos:
    causa_leg = diag["causa_legivel"]
    if causa_leg in causas_contagem:
        causas_contagem[causa_leg] += 1

# Se todos os 3 casos tiverem a mesma causa (raro mas possível),
# complementar com distribuição típica baseada na literatura
total_diag = sum(causas_contagem.values())
if total_diag < 3:
    # Distribuição típica: 40% param, 35% insuf, 25% sat
    causas_contagem = {
        "Alucinação Paramétrica": 5,
        "Contexto Insuficiente":  4,
        "Contexto Saturado":      3,
    }

# Usar dados simulados do DataFrame se causa_simulada disponível
if "causa_simulada" in df_baixo.columns:
    mapa_causa = {
        "alucinacao_parametrica": "Alucinação Paramétrica",
        "contexto_insuficiente":  "Contexto Insuficiente",
        "contexto_saturado":      "Contexto Saturado",
    }
    causas_df = df_baixo["causa_simulada"].dropna()
    causas_df = causas_df[causas_df != ""]
    if len(causas_df) >= 3:
        contagem_real = causas_df.map(mapa_causa).value_counts()
        for causa, cnt in contagem_real.items():
            if causa in causas_contagem:
                causas_contagem[causa] = int(cnt)

# ── Pie chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor="#F8F9FA")
fig.suptitle(
    "LAB 6 — Distribuição das Causas de Faithfulness < 0.70",
    fontsize=13, fontweight="bold", y=1.01
)

# -- Pie chart (esquerda) --
ax_pie = axes[0]
cores_causas = ["#4472C4", "#ED7D31", "#A9D18E"]
explode      = [0.05, 0.05, 0.05]
valores      = list(causas_contagem.values())
rotulos      = list(causas_contagem.keys())

wedges, texts, autotexts = ax_pie.pie(
    valores,
    labels=rotulos,
    autopct="%1.1f%%",
    startangle=140,
    colors=cores_causas,
    explode=explode,
    textprops={"fontsize": 10},
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontweight("bold")
    at.set_fontsize(11)
ax_pie.set_title("Proporção das causas\n(casos analisados)", fontsize=11, pad=12)

# -- Barras de contagem (direita) --
ax_bar = axes[1]
barras = ax_bar.bar(
    rotulos, valores,
    color=cores_causas, alpha=0.87,
    edgecolor="white", linewidth=0.7,
)
for barra, val in zip(barras, valores):
    ax_bar.text(
        barra.get_x() + barra.get_width()/2,
        barra.get_height() + 0.05,
        str(val),
        ha="center", va="bottom",
        fontsize=13, fontweight="bold"
    )
ax_bar.set_ylabel("Número de casos", fontsize=10)
ax_bar.set_title("Contagem absoluta por causa", fontsize=11)
ax_bar.tick_params(axis="x", labelsize=9)
ax_bar.set_ylim(0, max(valores) * 1.3)
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("distribuicao_causas_faithfulness.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✓ Gráfico salvo: distribuicao_causas_faithfulness.png")

## 12. Manual de Correção Rápida

Referência para uso operacional em produção de sistemas RAG jurídicos.

---

| Causa | Sintoma Diagnóstico | Solução Imediata |
|-------|---------------------|------------------|
| **Alucinação Paramétrica** | Afirmações na resposta sem correspondência em nenhum contexto recuperado; LLM usa dados de treinamento ("sabe-se que...", "estudos mostram...") | Aplicar prompt restritivo: *"Use APENAS os trechos abaixo. Se a informação não estiver nos trechos, informe que os documentos não são suficientes."* |
| **Contexto Insuficiente** | Context Recall baixo (< 0,60); ground truth contém informações ausentes em todos os contextos recuperados; resposta incompleta ou evasiva | Aumentar `k` para 10 ou 15; aplicar Contextual Retrieval (BM25 + kNN híbrido); usar query expansion com HyDE |
| **Contexto Saturado** | Context Precision baixa (< 0,55); muitos trechos recuperados com baixa relevância; resposta mistura normas de diferentes matérias | Aplicar reranking com CrossEncoder (BGE-Reranker-v2-m3); selecionar top-3 por relevância; filtrar por categoria jurídica do documento |

> **Regra de ouro:** Sempre verificar o tripé — Faithfulness + Context Recall + Context Precision — antes de diagnosticar. A causa raiz está na interação entre esses três scores.

In [ ]:
# ── Resumo final do LAB 6 ────────────────────────────────────
print("=" * 60)
print("           RESUMO FINAL — LAB 6")
print("=" * 60)
print(f"  Total de pares avaliados : {len(df_baseline)}")
print(f"  Com Faithfulness < {LIMIAR_FAITHFULNESS}   : {n_baixo} ({pct_baixo:.1f}%)")
print(f"  Casos diagnosticados      : {len(diagnosticos)}")
print("")
print("  Resultados das correções:")
for res in resultados_correcao:
    melhoria_pct = (res['melhoria'] / res['faith_antes']) * 100 if res['faith_antes'] > 0 else 0
    status = "✅" if res['faith_depois'] >= LIMIAR_FAITHFULNESS else "⚠"
    print(f"  {status} {res['causa']:<26} "
          f"{res['faith_antes']:.3f} → {res['faith_depois']:.3f} "
          f"({melhoria_pct:+.1f}%) [{res['correcao_aplicada']}]")
print("=" * 60)
print(f"  Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

## 13. Checklist de Conclusão — LAB 6

Marque cada item após verificação:

- ☐ **[C6.1]** Cabeçalho com objetivo, taxonomia das 3 causas e diagrama de decisão revisado
- ☐ **[C6.2]** Dependências instaladas sem erros (ragas, sentence-transformers, etc.)
- ☐ **[C6.3]** Variáveis de ambiente configuradas para o ambiente do lab
- ☐ **[C6.4]** Baseline carregado (ou simulado) e queries com Faithfulness < 0,70 filtradas
- ☐ **[C6.5]** Função `diagnosticar_query` executada nas 3 queries com score mais baixo
- ☐ **[C6.6]** Correção aplicada conforme a causa: prompt restritivo / k=10 / reranking top-3
- ☐ **[C6.7]** Faithfulness recalculado após correção e melhoria percentual calculada
- ☐ **[C6.8]** Gráficos exportados (barras horizontais + pie chart de distribuição de causas)